# 3. The same tests on another ensemble

Notebooks 1 and 2 step through the analysis for CMIP6. This notebook runs the same steps on any ensemble
(for example WIEMIP) using the packaged versions in `tcre_evidence.process_tests` and
`tcre_evidence.process_plots`. It is set up for the CMIP6 example so that it runs as it stands.

Inputs (details in `docs/new_ensemble.md`):

* a values CSV with a `model` column and one column per quantity (name the β/γ convention, e.g.
  `gamma_land_coubgc`);
* a flags CSV with a `model` column, one 0/1 column per process (e.g. `demography`) and, optionally, a
  `family` column.

In [ ]:
import numpy as np
import pandas as pd

from tcre_evidence import process_plots as pp, process_tests as pt
from tcre_evidence.paths import DATA

pd.set_option("display.precision", 2)

NAME = "cmip6"                    # name of this ensemble's cache, e.g. "wiemip"
VALUES = DATA / "examples" / "cmip6_land_feedbacks.csv"
FLAGS = DATA / "examples" / "cmip6_process_flags.csv"
QUANTITY = "gamma_land_coubgc"    # a column of VALUES
KIND = "additive"                 # "additive" for gamma (PgC/K), "multiplicative" for beta (PgC/ppm)
LABEL = "γ_L (PgC/K)"

In [ ]:
y, flags, family = pt.load_ensemble(VALUES, FLAGS, QUANTITY)
pp.by_flag(y, flags, family, LABEL);

In [ ]:
yy = np.log(y) if KIND == "multiplicative" else y
b = pt.flag_budget(yy, flags)
print(f"flags explain {b['r2']:.0%} of the spread; each model predicted from the others: {b['loo_q2']:.2f}")
pp.power(pt.detection_power(yy, flags, {p: 0.5 * yy.std() for p in flags.columns}));

The Bayesian tests need sampling. Run the command printed below in a terminal (a few minutes with
`--quick`, about an hour without), then run the remaining cells.

In [ ]:
print(f"python scripts/precompute.py --name {NAME} --values {VALUES} --flags {FLAGS} "
      f"--quantities {QUANTITY}:{KIND} --jobs 4")

In [ ]:
res = pp.load(NAME, QUANTITY)
pp.prior_posterior(res);

In [ ]:
pp.recovery(res);
pt.evidence_table(res["prior_vs_posterior"], res["synthetic_effects_summary"],
                  res["synthetic_no_effects_summary"], res["power"])

In [ ]:
pp.loo(res, LABEL);
res["loo_summary"]

Before quoting a number: `rhat_max` below 1.01 and few or no divergences, no `--quick` runs.
`res["run"]` records the priors, effect sizes, settings and package versions used.

In [ ]:
res["run"]["diagnostics"]